In [20]:
#Création d’un nouveau agent sans System Message
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

# Initialiser le modèle Ollama
model = ChatOllama(
model="llama3.2", # ou mistral, gemma, etc.
temperature=0
)
agent = create_agent(model=model)
question = HumanMessage(content="Quelle est la capitale de la lune ?")
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

La question classique !

Il n'y a pas de capitale sur la Lune, car elle n'est pas habitée par des êtres humains ou d'autres espèces. La Lune est un satellite naturel de la Terre et n'a pas de gouvernement, de population ou de structure politique.

Cependant, il y a eu plusieurs missions spatiales qui ont envoyé des robots sur la surface de la Lune, notamment les missions Apollo américaines dans les années 1960 et 1970. Les astronautes qui ont visité la Lune ont déposé des drapeaux et des symboles nationaux sur sa surface, mais cela n'a pas créé une capitale.

En résumé, il n'y a pas de capitale sur la Lune, car elle est un corps céleste inhabité et non soumis à une autorité politique.


In [21]:
#Agent avec System Message : un message de contrôle qui définit le comportement global du modèle

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Charger les variables d'environnement (.env)
load_dotenv(override=True)


# Initialiser le modèle Ollama
model = ChatOpenAI(model="gpt-5.2", # ou mistral, gemma, etc.
temperature=1
)

question = HumanMessage(content="Quelle est la capitale de la lune ?")

system_prompt = "Vous êtes un auteur de science-fiction ; créez une capitale à la demande des utilisateurs."
scifi_agent = create_agent(
model=model,
system_prompt=system_prompt
)
response = scifi_agent.invoke(
{"messages": [question]}
)
print(response['messages'][-1].content)

La Lune n’a pas de capitale au sens politique réel : elle n’appartient à aucun pays et ne possède pas d’État lunaire reconnu.

Mais si tu me le demandes en mode science-fiction, voici **une capitale lunaire** plausible :

## **Sélénépolis**
- **Lieu** : au bord du cratère **Shackleton**, près du pôle Sud lunaire (là où l’on peut capter de la lumière presque en continu et accéder à la glace).
- **Statut** : capitale administrative de la **Confédération des Habitats Lunaires**, née quand les premières bases se sont reliées par tunnels pressurisés.
- **Architecture** : une ville **semi-enterrée**, faite de dômes bas recouverts de régolithe (contre les radiations), reliés par des galeries et des ascenseurs à vis.
- **Centre symbolique** : la **Place du Terminator**, un immense atrium où l’on voit la bande de clair-obscur glisser lentement sur l’horizon.
- **Pouvoir & économie** : gouvernée par un **Conseil des Quatorze Modules** (chaque module représente une “cité-caverne”), et prospère gr

In [22]:
#Agent avec Few-shot learning : une méthode où le modèle
#apprend une nouvelle tâche ou classe à partir de quelques
#exemples seulement

system_prompt = """

Vous êtes un auteur de science-fiction et vous devez créer une capitale spa-
tiale à la demande d'un utilisateur.

Utilisateur : Quelle est la capitale de Mars ?
Auteur : Marsialis
Utilisateur : Quelle est la capitale de Vénus ?
Auteur : Venusovia
"""

scifi_agent = create_agent(
model=model,
system_prompt=system_prompt
)
response = scifi_agent.invoke(
{"messages": [question]}
)
print(response['messages'][-1].content)

Lunarisia


In [23]:
# test 
question2= HumanMessage(content="Quelle est la capitale de la polognie ?")
response = scifi_agent.invoke(
{"messages": [question2]}
)
print(response['messages'][-1].content)

Polognisbourg


In [24]:
#Agent avec réponse structurée: une sortie organisée selon un
#format prédéfini, plutôt que du texte libre.
system_prompt = """

Vous êtes un auteur de science-fiction et vous devez créer une capitale spa-
tiale à la demande d'un utilisateur.

Veuillez respecter la structure ci-dessous.
Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Économie : Principaux secteurs d'activité
"""
question = HumanMessage(content="Quelle est la capitale de la lune ?")
scifi_agent = create_agent(
model=model,
system_prompt=system_prompt
)
response = scifi_agent.invoke(
{"messages": [question]}
)
print(response['messages'][-1].content)

Nom : Sélénopolis  
Localisation : Mer de la Tranquillité, sous un dôme pressurisé relié à un réseau de tunnels basaltiques  
Ambiance : Silencieuse, lumineuse, solennelle  
Économie : Extraction d’hélium-3, fabrication en faible gravité, recherche scientifique et diplomatie intercoloniale


In [25]:
#Agent avec réponse structurée en utilisant BaseModel : rendre la
#réponse facile à exploiter automatiquement par un programme ou
#un système.

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

# Initialiser le modèle Ollama
model = ChatOllama(
model="llama3.2", # ou mistral, gemma, etc.
temperature=1
)

class CapitalInfo(BaseModel):
    nom:str
    Localisation:str
    Ambiance:str
    Economie:str

system_prompt = """

Vous êtes un auteur de science-fiction et vous devez créer une capitale spa-
tiale à la demande d'un utilisateur.

Veuillez respecter la structure ci-dessous.
Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Économie : Principaux secteurs d'activité
"""
agent=create_agent(model=model,system_prompt=system_prompt, response_format=CapitalInfo)    

question = HumanMessage(content="Quelle est la capitale de la lune ?")
response = agent.invoke(
{"messages": [question]}
)
response_structure=response["structured_response"]
print(response_structure)


nom='Capitole Lunaire' Localisation='Lun' Ambiance='Lunaïque' Economie='Tourisme Spatiale'


In [28]:
from langchain.tools import tool
from langchain.agents import create_agent
@tool("meteo_capitale")
def meteo_capitale(ville: str) -> str:
    """
    Donne la météo d'une capitale (valeurs fixes pour test).
    Args:
    ville: nom de la capitale
    """
    print("tool meteo_capitale utilisé")
    temperature = 25
    humidite = 60
    pression = 1013
    return (
    f"Météo à {ville} : "
    f"Température = {temperature}°C, "
    f"Humidité = {humidite}%, "
    f"Pression = {pression} hPa"
    )

system_prompt = "Utilises les tools pour répondre aux questions"
agent = create_agent(
model=model,
tools=[meteo_capitale],
system_prompt=system_prompt,
)
question = HumanMessage(content="Quelle est la météo à Capitale lunaire ?")
response = agent.invoke(
{"messages": [question]}
)
print(response['messages'][-1].content)

tool meteo_capitale utilisé
Voici la réponse formatée :

La météo actuelle à Capitale lunaire est la suivante :
- Température : 25°C
- Humidité : 60%
- Pression : 1013 hPa


In [ ]:
#Agent avec Web Search Tool
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv
load_dotenv(override=True)

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Recherche des informations sur le web via Tavily."""
    return tavily_client.search(query)

#Teste si l'outil fonctionne correctement
web_search.invoke("Qui est le Président de commune actuel de Marrakech ?")

#Sortie de l’agent
agent = create_agent(
model=model,
tools=[web_search]
)
question = HumanMessage(content="Qui est le Président de commune actuel de Marrakech ?")
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)



Le président actuel de la commune de Marrakech est Fatima Zahra Mansouri du Parti Authenticité et Modernité (PAM).


In [ ]:
#Agent sans mémoire
agent = create_agent(model= model)

question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")

response = agent.invoke({"messages": [question]})

print(response['messages'][-1].content)


Bonjour Sami ! Enchanté de faire votre connaissance. Je suis ravi de vous aider dans vos projets de développement, qu'il s'agisse d'applications, de jeux, de systèmes d'exploitation ou de tout autre projet technologique. N'hésitez pas à me poser vos questions et à partager vos problèmes, je ferai de mon mieux pour vous aider.

Qu'est-ce que vous travaillez en ce moment ? Avez-vous un projet spécifique sur lequel vous avez besoin d'aide ou de conseils ?


In [38]:
question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

Déterminer son métier peut être un processus complexe et personnel. Pour vous aider, voici quelques étapes que vous pouvez suivre :

1. **Identifiez vos compétences** : Faites une liste des compétences que vous possédez, quelles sont vos forces et vos faiblesses. Quels sont vos domaines d'expertise ?

2. **Définissez vos objectifs** : Qu'est-ce qui vous motive ? Que cherchez-vous dans votre carrière ? Les revenus ? L'équilibre entre travail et vie personnelle ? La possibilité de travailler à distance ?

3. **Explorez vos intérêts** : Quels sont les domaines d'activité qui vous intéressent le plus ? Qu'est-ce que vous aimez faire dans votre temps libre ?

4. **Pensez à vos expériences passées** : Quelles ont été vos expériences professionnelles précédentes ? Quels sont les défis que vous avez rencontrés et comment les avez-vous surmontés ?

5. **Recherchez des métiers qui correspondent** : Utilisez les outils en ligne tels que les sites de recherche de travail, les forums professionnels

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
model=model,
checkpointer=InMemorySaver(),
)

question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({"messages": [question]},config,)
question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke({"messages": [question]},config,)
print(response['messages'][-1].content)

Vous êtes un développeur ! C'est un métier très passionnant qui implique la conception, le développement et la maintenance de logiciels et d'applications numériques. Vous travaillez souvent avec des langages de programmation tels que Java, Python, JavaScript ou C++, et vous utilisez des frameworks tels que React, Angular ou Vue.js.

En tant que développeur, vous avez la possibilité de travailler sur des projets variés, allant de la création d'applications web à la développement d'application mobile. Vous devez également être en mesure de résoudre des problèmes complexes et de collaborer avec d'autres équipes pour atteindre les objectifs du projet.

Qu'est-ce que vous aimez le plus dans votre métier de développeur ?
